# 🤖 Workshop: Bau deinen eigenen WM-Experten 2026
## Notebook 4: Bau deines Fußball-Experten-Agenten

Ein **KI-Agent** ist mehr als ein einfacher Chatbot. Er kann selbstständig entscheiden, welche Aktionen er ausführt. Er überlegt, welche Informationen er braucht, ruft die passenden Tools oder Datenbanken ab und formuliert daraus eine schlüssige Antwort.

In diesem finalen Notebook bauen wir einen voll funktionsfähigen **ReAct-Agenten** (Reasoning & Action). Der Agent kann selbst entscheiden, ob er in unserer RAG-Datenbank nachliest (für Trivia und Spielpläne) oder den MCP-Server fragt (für ELO-Ratings und Wahrscheinlichkeiten).

### Lernziele:
1. Aufbau eines Agenten-Loops mit Funktions-Aufrufen (Function Calling).
2. Verknüpfung von RAG (Wissensdatenbank) und Live-Tools (MCP).
3. Testen des Agenten mit komplexen Fragestellungen.


### Schritt 1: Initialisierung
Wir verbinden uns mit ChromaDB (wo die fertige Collection `wm2026` der Pipeline liegt) und stellen die Clients bereit.


In [ ]:
import os
import json
import httpx
import chromadb
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://litellm.fh-swf.cloud/v1"),
    api_key=os.environ.get("OPENAI_API_KEY", "dummy")
)

CHROMA_HOST = os.environ.get("CHROMA_HOST", "chromadb")
CHROMA_PORT = int(os.environ.get("CHROMA_PORT", 8000))
MCP_URL = "https://wm2026.fh-swf.cloud/mcp"

try:
    chroma_client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
    # Wir nutzen die von der Pipeline befüllte Haupt-Collection
    collection = chroma_client.get_collection("wm2026")
    print("✅ Erfolgreich mit globaler Vektordatenbank verbunden.")
except Exception as e:
    print("❌ Verbindung zur globalen ChromaDB fehlgeschlagen. Nutze Fallback...", e)
    chroma_client = chromadb.PersistentClient(path="../data-pipeline/data/chroma")
    collection = chroma_client.get_collection("wm2026")


### Schritt 2: Hilfsfunktionen für Tools & RAG
Wir definieren die Aktionen, die unser Agent ausführen kann.


In [ ]:
def query_rag_database(query: str) -> str:
    """Sucht in der Vektordatenbank nach historischen Fakten."""
    query_emb = client.embeddings.create(
        model="text-embedding-3-small",
        input=[query]
    ).data[0].embedding
    
    results = collection.query(query_embeddings=[query_emb], n_results=3)
    docs = results["documents"][0]
    return "\n\n".join(docs)

def call_mcp_tool(tool_name: str, parameters: dict) -> str:
    """Ruft das passende Tool vom MCP-Server ab."""
    response = httpx.post(f"{MCP_URL}/call", json={"tool": tool_name, "parameters": parameters})
    if response.status_code != 200:
        return f"Fehler im Tool: {response.text}"
    result = response.json().get("result", {})
    return json.dumps(result, ensure_ascii=False)

# Teste die RAG-Suche
print(query_rag_database("Wer ist Rekordweltmeister?"))


### Schritt 3: Tool-Definitionen für das LLM
Wir beschreiben dem LLM die verfügbaren Funktionen im OpenAI-Format. Das LLM entscheidet anhand der Beschreibungen, welches Tool es nutzen möchte.


In [ ]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "query_rag_database",
            "description": "Sucht in der WM-Datenbank nach Fakten, Historie, Trivia und Spielort-Infos.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Suchanfrage"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_team_elo",
            "description": "Gibt ELO-Rating, Weltrang und historische Statistiken zu einem Team zurück.",
            "parameters": {
                "type": "object",
                "properties": {
                    "team": {"type": "string", "description": "Teamname (z.B. Deutschland)"}
                },
                "required": ["team"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "simulate_tournament",
            "description": "Simuliert das WM-Turnier per Monte-Carlo. Gibt die Chancen für das Team an.",
            "parameters": {
                "type": "object",
                "properties": {
                    "team": {"type": "string", "description": "Teamname für Detailbericht"},
                    "n_sims": {"type": "integer", "description": "Anzahl Simulationen (Standard: 1000)"}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_news",
            "description": "Sucht aktuelle Nachrichten zur WM 2026 über DuckDuckGo.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Suchanfrage"}
                },
                "required": ["query"]
            }
        }
    }
]


### Schritt 4: Der Agenten-Loop (ReAct)
Wir implementieren den Loop. Wenn das Sprachmodell einen Tool-Aufruf anfordert, führen wir diesen aus, hängen das Ergebnis an den Verlauf an und rufen das Modell erneut auf.


In [ ]:
def run_agent(question: str) -> str:
    print(f"Frage: {question}")
    messages = [
        {
            "role": "system",
            "content": "Du bist ein fußballbegeisterter WM-Experten-Agent. Beantworte alle Fragen präzise und faktenbasiert auf Deutsch. Nutze deine Tools! Wenn eine Frage historische Infos betrifft, nutze query_rag_database. Wenn es um ELO-Ratings oder Wahrscheinlichkeiten geht, nutze get_team_elo oder simulate_tournament."
        },
        {"role": "user", "content": question}
    ]
    
    # 1. Anfrage ans LLM
    response = client.chat.completions.create(
        model="gpt-4o",  # LiteLLM leitet dies an das konfigurierte Modell weiter
        messages=messages,
        tools=tools_schema,
        tool_choice="auto"
    )
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls
    
    if tool_calls:
        print("🤖 Agent überlegt und entscheidet sich für Tool-Nutzung...")
        messages.append(response_message)
        
        for tool_call in tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            
            print(f"  → Tool aufrufen: '{func_name}' mit Parameter {func_args}")
            
            if func_name == "query_rag_database":
                result_str = query_rag_database(func_args["query"])
            else:
                result_str = call_mcp_tool(func_name, func_args)
                
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": func_name,
                "content": result_str
            })
        
        # 2. Zweite Anfrage ans LLM mit den Tool-Ergebnissen
        final_response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages
        )
        return final_response.choices[0].message.content
    else:
        return response_message.content


### Schritt 5: Agenten testen!
Stellen wir unserem Agenten nun eine komplexe Frage, die sowohl statisches Wissen als auch aktuelle Simulationen benötigt.


In [ ]:
agenten_antwort = run_agent(
    "Wer spielt in Gruppe E und wie hoch ist die Wahrscheinlichkeit, dass der ELO-Favorit dieser Gruppe die WM gewinnt?"
)
print("\n=== Antwort des Agenten ===")
print(agenten_antwort)


### 🏆 Herzlichen Glückwunsch!
Du hast erfolgreich einen KI-Agenten gebaut, der RAG und Live-Tools kombiniert, um echte Fußball-Prognosen zu erstellen! Du kannst nun im Chat oder in JupyterLab weitere Fragen ausprobieren.
